In [1]:
import networkx as nx
import pandas as pd
import pickle
import time
import numpy as np
import os
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import deque
from torch_geometric.nn import SAGEConv, LayerNorm
from torch_geometric.data import Data, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [2]:
# GLOBAL CONFIGURATION
# For Local Jupyter
DATA_FOLDER = "graph_data" 
# DATA_FOLDER = "/content/drive/MyDrive/graph_data" (for google collab)

if not os.path.exists(DATA_FOLDER):
    os.makedirs(DATA_FOLDER)

def get_path(key, file_type):
    suffixes = {
        'edges': f"{key}_edges",
        'labels': f"{key}_edge_labels",
        'regress': f"{key}_edge_regress_val",
        'graphset': "full_graphset.pkl"
    }
    return os.path.join(DATA_FOLDER, suffixes[file_type])

print(f"Path Manager initialized. Data directory: {os.path.abspath(DATA_FOLDER)}")

Path Manager initialized. Data directory: /Users/harshita/Desktop/venv/graph_data


In [3]:
def compute_stress_centrality(G):
    nodes = list(G.nodes())
    stress = {v: 0.0 for v in nodes}
    for s in nodes:
        S = []; P = {v: [] for v in nodes}; sigma = {v: 0.0 for v in nodes}
        sigma[s] = 1.0; dist = {v: -1 for v in nodes}; dist[s] = 0
        queue = deque([s])
        while queue:
            v = queue.popleft()
            S.append(v)
            for w in G.neighbors(v):
                if dist[w] < 0:
                    dist[w] = dist[v] + 1
                    queue.append(w)
                if dist[w] == dist[v] + 1:
                    sigma[w] += sigma[v]
                    P[w].append(v)
        delta = {v: 0.0 for v in nodes}
        while S:
            w = S.pop()
            for v in P[w]:
                delta[v] += (delta[w] + 1)
            if w != s:
                stress[w] += delta[w] * sigma[w]
    return stress

In [6]:
def genDataset(key, graph_set):
    G = graph_set[key]
    base_sc_dict = compute_stress_centrality(G)
    base_sc_max = max(base_sc_dict.values())
    
    print(f"Processing {key} | Max SC: {base_sc_max}")
    st = time.time()
    edge_table = {}
    non_edges = list(nx.non_edges(G))

    for i, (u, v) in enumerate(non_edges):
        G.add_edge(u, v)
        new_sc = compute_stress_centrality(G)
        edge_table[(u,v)] = base_sc_max - max(new_sc.values())
        G.remove_edge(u, v)
        

    with open(get_path(key, 'edges'), 'wb') as file:
        pickle.dump(edge_table, file)
    return time.time() - st


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

graphset = {}

print("Starting Full Dataset Generation (40 Graphs)")

#  10 Barabasi-Albert graphs 
for i in range(10):
    key = f'ba{i}'
    graphset[key] = nx.barabasi_albert_graph(200, 4, seed=SEED+i)
    genDataset(key, graphset)
    print(f"Completed: {key}")

# 10 Erdos-Renyi graphs
for i in range(10):
    key = f'er{i}'
    graphset[key] = nx.erdos_renyi_graph(200, 0.05, seed=SEED+i+100)
    genDataset(key, graphset)
    print(f"Completed: {key}")

# 10 Path graphs
for i in range(10):
    key = f'path{i}'
    graphset[key] = nx.path_graph(200)
    genDataset(key, graphset)
    print(f"Completed: {key}")

# 10 Tree graphs (random labeled trees)
for i in range(10):
    key = f'tree{i}'
    graphset[key] = nx.random_labeled_tree(200, seed=SEED+i+200)
    genDataset(key, graphset)
    print(f"Completed: {key}")

# 3. Save the final graphset dictionary
graphset_save_path = get_path(None, 'graphset')
with open(graphset_save_path, 'wb') as f:
    pickle.dump(graphset, f)

print(f"\nSUCCESS: All 40 graphs and their datasets are stored in '{DATA_FOLDER}'")

Starting Full Dataset Generation (40 Graphs)
Processing ba0 | Max SC: 41182.0
Completed: ba0
Processing ba1 | Max SC: 23182.0
Completed: ba1
Processing ba2 | Max SC: 32902.0
Completed: ba2
Processing ba3 | Max SC: 29716.0
Completed: ba3
Processing ba4 | Max SC: 23784.0
Completed: ba4
Processing ba5 | Max SC: 20038.0
Completed: ba5
Processing ba6 | Max SC: 28984.0
Completed: ba6
Processing ba7 | Max SC: 27746.0
Completed: ba7
Processing ba8 | Max SC: 30398.0
Completed: ba8
Processing ba9 | Max SC: 31380.0
Completed: ba9
Processing er0 | Max SC: 4492.0
Completed: er0
Processing er1 | Max SC: 4832.0
Completed: er1
Processing er2 | Max SC: 4622.0


In [ ]:
key = 'ba0' 

file_path = get_path(key, 'edges')

with open(file_path, 'rb') as f:
    edge_sc = pickle.load(f)

# Sort the items (Edge Tuple, Reduction Value) in descending order
dummy = sorted(edge_sc.items(), key=lambda x: x[1], reverse=True)

print(f"Loaded and sorted {len(dummy)} edges for {key}.")
print(f"Highest reduction edge: {dummy[0][0]} with a drop of {dummy[0][1]:.2f}")

In [ ]:
x = [i for i in range(len(dummy))]
y = []
for edge, val in dummy:
    y.append(val)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.set(style="darkgrid")
plt.figure(figsize=(7, 5))
sns.scatterplot(x=x, y=y)

# Add title and labels
title = "BA Graph " + key + " (n=200, m=4)"
plt.title(title)
plt.xlabel('Index')
plt.ylabel('Reductions')

# Show the plot
plt.show()

In [ ]:
def gen_labels(graphset, k=1500, seed=42):
    np.random.seed(seed)
    
    for key in graphset:
        with open(get_path(key, 'edges'), 'rb') as f:
            edge_sc = pickle.load(f)
        dummy = sorted(edge_sc.items(), key = lambda x: x[1], reverse=True)
        numbers = len(dummy)
        actual_k = min(k, numbers)
        weights = (np.linspace(-1, 1, numbers))**4
        weights /= weights.sum()
        indices = np.sort(np.random.choice(numbers, size=actual_k, replace=False, p=weights))
        
        edge_labels = {1: [], 0: []}
        edge_regress_val = []
        for i in indices:
            edge, val = dummy[i]
            if val > 0: edge_labels[1].append(edge)
            else: edge_labels[0].append(edge)
            edge_regress_val.append(val)
            
        with open(get_path(key, 'labels'), 'wb') as f: pickle.dump(edge_labels, f)
        with open(get_path(key, 'regress'), 'wb') as f: pickle.dump(edge_regress_val, f)
        print(f"Labels saved for {key}")


print("\n Generating Sampled Labels for Training ")
gen_labels(graphset, k=1500)

In [ ]:
def load_files(key):
    
    with open(get_path(key, 'edges'), 'rb') as f: edge_max_sc = pickle.load(f)
        
    dummy = sorted(edge_max_sc.items(), key = lambda x: x[1], reverse=True)
    
    with open(get_path(key, 'labels'), 'rb') as f: edge_labels = pickle.load(f)
        
    with open(get_path(key, 'regress'), 'rb') as f: edge_regress = pickle.load(f)
        
    return edge_max_sc, dummy, edge_labels, edge_regress

In [ ]:
def plot_diff(graphset_subset):
    n_graphs = len(graphset_subset)

    if n_graphs == 0:
        print("No graphs to plot.")
        return

    ncols = 2 if n_graphs > 1 else 1
    nrows = (n_graphs + ncols - 1) // ncols

    fig, axs = plt.subplots(nrows, ncols, figsize=(12, 5 * nrows))
    
    # Ensure axs is always an array even if there's only 1 plot
    if n_graphs == 1:
        axs = [axs]
    else:
        axs = axs.flatten()

    for k, key in enumerate(graphset_subset):
        # Use the load_files function we defined earlier
        _, sorted_edges, _, _ = load_files(key)
        
        # Extract the reduction values 
        reductions = [val for edge, val in sorted_edges]
        
        axs[k].plot(reductions, color='blue', linewidth=1.5)
        axs[k].set_title(f"Graph: {key} ({nx.number_of_nodes(graphset_subset[key])} nodes)")
        axs[k].set_xlabel("Ranked Edges (Best to Worst)")
        axs[k].set_ylabel("Stress Reduction")
        axs[k].grid(True, linestyle='--', alpha=0.6)

    # Hide any unused subplots (if n_graphs is odd)
    for j in range(k + 1, len(axs)):
        fig.delaxes(axs[j])

    plt.tight_layout()
    plt.show()


keys_to_plot = ['ba0', 'er0', 'path0', 'tree0']
subset = {k: graphset[k] for k in keys_to_plot if k in graphset}

plot_diff(subset)

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import torch
import numpy as np
import networkx as nx

def features(G):
    # Degree Centrality
    x_deg = nx.degree_centrality(G)
    x_deg = np.array(list(x_deg.values())).reshape(-1, 1)

    # Eigenvector Centrality
    x_evc_dict = nx.eigenvector_centrality_numpy(G)
    x_evc = np.array(list(x_evc_dict.values())).reshape(-1, 1)

    # Closeness Centrality
    x_cc = nx.closeness_centrality(G)
    x_cc = np.array(list(x_cc.values())).reshape(-1, 1)

    # PageRank
    x_pr = nx.pagerank(G)
    x_pr = np.array(list(x_pr.values())).reshape(-1, 1)

    # Clustering Coefficient
    x_clc = nx.clustering(G)
    x_clc = np.array(list(x_clc.values())).reshape(-1, 1)

    # Harmonic Centrality
    x_hc = nx.harmonic_centrality(G)
    x_hc = np.array(list(x_hc.values())).reshape(-1, 1)

    # Horizontal Stack all features: Shape (N, 6)
    x_combined = np.hstack((x_deg, x_evc, x_cc, x_pr, x_clc, x_hc))
    
    # Normalize features to [0, 1] range
    scaler = MinMaxScaler()
    x_scaled = scaler.fit_transform(x_combined)
    
    # Convert to PyTorch Tensor for GNN processing
    node_features = torch.tensor(x_scaled, dtype=torch.float32)

    return node_features

In [ ]:
def split_to_batches(graph_data, batch_size, shuffle=True):
    
    x = graph_data.x
    edge_index = graph_data.edge_index
    y = graph_data.y
    
    num_batches = len(y) // batch_size
    data_objects = []
    
    for i in range(num_batches):
        # Slice batch of edge targets
        y_batch = y[i*batch_size : (i+1)*batch_size]
        
        # Create new Data object
        new_data = Data(x=x, edge_index=edge_index, y=y_batch)
        data_objects.append(new_data)
        
    # Handle the remaining samples
    if len(y) % batch_size != 0:
        y_batch = y[num_batches*batch_size:]
        new_data = Data(x=x, edge_index=edge_index, y=y_batch)
        data_objects.append(new_data)
        
    return data_objects

In [ ]:
import random
import torch
import numpy as np
from torch_geometric.data import Data, DataLoader
from sklearn.model_selection import train_test_split

def data_splitting(graphset, batch_size):
    
    random.seed(55)
    key_list = list(graphset.keys())
    random.shuffle(key_list)
    
    # Graph-wise Split (80% for training,validation; 20% for final test)
    split_point = int(len(graphset) * 0.2)
    test_keys = key_list[:split_point]
    train_keys = key_list[split_point:]
    
    test_set = {key: graphset[key] for key in test_keys}
    train_set = {key: graphset[key] for key in train_keys}
    
    print(f"Total Graphs: {len(graphset)}")
    print(f"Graphs assigned to Training/Val: {list(train_set.keys())}")
    print(f"Graphs held out for Final Test: {list(test_set.keys())}")

    data_objects = []

    for key in train_set:
        G = train_set[key]
        
        # Extract Node Features
        node_features = features(G)
        
        # Get Graph Topology
        edge_index = torch.tensor(list(G.edges), dtype=torch.long).T

        # Load Sampled Labels and Regression Values
        _, _, edge_labels, edge_regress = load_files(key)
        
        # Construct the Target Matrix (y)
        # [Node_U, Node_V, Class_Label, Regression_Value]
        edges_dataset = np.array(edge_labels[1] + edge_labels[0])
        l1 = np.ones((len(edge_labels[1]), 1))
        l0 = np.zeros((len(edge_labels[0]), 1))
        
        Y_classes = np.vstack((l1, l0))
        Y_regress_values = np.array(edge_regress).reshape(-1, 1)
        
        # Horizontal stack to create a matrix
        y_combined = np.hstack((edges_dataset, Y_classes, Y_regress_values))
        
        # Shuffle indices within the graph
        np.random.seed(212)
        np.random.shuffle(y_combined)
        y_tensor = torch.tensor(y_combined, dtype=torch.float32)

        # Create the PyG Data object
        graph_data = Data(x=node_features, edge_index=edge_index, y=y_tensor)
        
        # Split into manageable chunks
        chunks = split_to_batches(graph_data, batch_size)
        data_objects.extend(chunks)

    # Global shuffle of all chunks from all training graphs
    random.shuffle(data_objects)
    
    
    #  20% of training data for validation
    train_data, val_data = train_test_split(data_objects, test_size=0.2, random_state=42, shuffle=True)
    
    # Create DataLoaders
    # batch_size=1 because each element in the list is already a pre-batched chunk of edges
    train_loader = DataLoader(train_data, batch_size=1, shuffle=True) 
    val_loader = DataLoader(val_data, batch_size=1, shuffle=False)

    print(f"\nData Pipeline Ready:")
    print(f"Created {len(data_objects)} total edge-chunks.")
    print(f"Chunks in Training: {len(train_data)}")
    print(f"Chunks in Validation: {len(val_data)}")

    return train_loader, train_data, val_loader, val_data, train_set, test_set

In [ ]:
batch_size = 256 

train_loader, train_data, val_loader, val_data, train_set, test_set = data_splitting(graphset, batch_size)

print("\nDATA LOADER ANALYSIS\n")


print(f"Total Training Chunks:    {len(train_loader)}")
print(f"Total Validation Chunks:  {len(val_loader)}")
print(f"Hidden Test Graphs:       {len(test_set)} ({list(test_set.keys())})")

sample_batch = train_data[0]
print(f"\nSample Batch structure: {sample_batch}")
print(f"Node Feature Matrix (x) shape: {sample_batch.x.shape}") 
print(f"Edge Index shape:              {sample_batch.edge_index.shape}")
print(f"Target Info (y) shape:         {sample_batch.y.shape}")


In [ ]:
# take one batch from the training loader
example_batch = next(iter(train_loader))

print(f"Node Feature Matrix (x) Shape:  {example_batch.x.shape}") 
print(f"Adjacency List (edge_index):    {example_batch.edge_index.shape}")
print(f"Target Task Matrix (y) Shape:   {example_batch.y.shape}") 
print("-" * 40)

# Display the first 3 candidate edges in this batch
# These are the pairs the model will rank
for i in range(3):
    u = example_batch.y[i, 0].item()
    v = example_batch.y[i, 1].item()
    label = example_batch.y[i, 2].item()
    reduction = example_batch.y[i, 3].item()
    
    status = "REDUCES STRESS" if label == 1.0 else "NO REDUCTION"
    
    print(f"Candidate {i}: Edge ({int(u)}, {int(v)})")
    print(f"   Classification: {label} ({status})")
    print(f"   Regression:     {reduction:.4f} (Actual drop in Max SC)")
    print("\n")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv

class StressMinimizationGNN(nn.Module):
    def __init__(self, in_channels, hidden_channels=32, out_channels=64):
        super(StressMinimizationGNN, self).__init__()
        
        # GNN Layers
        self.sage1 = SAGEConv(in_channels, hidden_channels)
        self.sage2 = SAGEConv(hidden_channels, hidden_channels)
        self.sage3 = SAGEConv(hidden_channels, out_channels)
        
        # MLP 1 - Classification
        self.mlp_classifier = nn.Sequential(
            nn.Linear(2 * out_channels, 64), # Increased slightly to match capacity
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
        # MLP 2 - Ranking/Regression
        self.mlp_ranker = nn.Sequential(
            nn.Linear(2 * out_channels, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x, edge_index, edge_list):
        # 1. Generate Node Embeddings
        h = F.relu(self.sage1(x, edge_index))
        h = F.relu(self.sage2(h, edge_index))
        h = self.sage3(h, edge_index) # Output shape: [Num_Nodes, out_channels]
        
        # 2. Map indices for the candidate edges
        u_idx = edge_list[:, 0].long()
        v_idx = edge_list[:, 1].long()
        
        # 3. Create Edge Representations (Concat node u and node v)
        # [Num_Edges_in_Batch, 2 * out_channels]
        edge_repr = torch.cat([h[u_idx], h[v_idx]], dim=-1) 
        
        # 4. Final Predictions
        probs = self.mlp_classifier(edge_repr).squeeze(-1)
        scores = self.mlp_ranker(edge_repr).squeeze(-1)
        
        return probs, scores, h

In [ ]:
# Initialize the model with the 6 structural features we defined
test_model = StressMinimizationGNN(in_channels=6)
test_model.eval() # Set to evaluation mode

# Extract data from the example batch
x = example_batch.x
edges = example_batch.edge_index

# Extract the (u, v) pairs we want to predict
target_edges = example_batch.y[:, 0:2] 

# Perform the forward pass without calculating gradients
with torch.no_grad():
    probs, scores, h_nodes = test_model(x, edges, target_edges)

# Analysis Outputs
print("\nMODEL OUTPUT ANALYSIS\n")
print(f"Node Embedding Matrix Shape: {h_nodes.shape}") 
print(f"Classification Output Shape: {probs.shape}")  
print(f"Ranking/Score Output Shape:  {scores.shape}") 

print(f"Max Probability: {probs.max().item():.4f}")
print(f"Min Probability: {probs.min().item():.4f}")


In [ ]:
class RankingLoss(nn.Module):
    def __init__(self, margin=1.0):
        super(RankingLoss, self).__init__()
        self.margin = margin

    def forward(self, s_pred, s_gt):
        # Sample three random permutations to create pairs
        n = s_pred.size(0)
        perm1 = torch.randperm(n)
        perm2 = torch.randperm(n)
        perm3 = torch.randperm(n)
        
        def compute_pair_loss(s_i, s_j, g_i, g_j):
            # Calculate the target direction: 1 if i > j, -1 if j > i, 0 if tie
            target = torch.sign(g_i - g_j)
            
            # If target is 0 (tie), the loss for that pair will be 0
            # Otherwise, use the standard ranking hinge loss formula
            return torch.relu(-target * (s_i - s_j) + self.margin) * (target != 0).float()

        # Sum the losses for the sampled pairs
        loss = compute_pair_loss(s_pred[perm1], s_pred[perm2], s_gt[perm1], s_gt[perm2]) + \
               compute_pair_loss(s_pred[perm2], s_pred[perm3], s_gt[perm2], s_gt[perm3]) + \
               compute_pair_loss(s_pred[perm3], s_pred[perm1], s_gt[perm3], s_gt[perm1])
        
        return loss.mean()

In [ ]:
# take one sample from the loader 
example_batch = next(iter(train_loader)) 

# Initialize model
test_model = StressMinimizationGNN(in_channels=6)
test_model.eval() # Set to eval mode for analysis

# Extract the target edge pairs 
example_edges = example_batch.y[:, 0:2] 

# Perform the test pass
with torch.no_grad():
    probs, scores, node_embeddings = test_model(
        example_batch.x, 
        example_batch.edge_index, 
        example_edges
    )

print("\n")
print(" DATA FLOW ANALYSIS ")
print("\n")
print(f"1. Input Node Features: {example_batch.x.shape}")
print(f"   (Nodes: {example_batch.x.size(0)}, Features: {example_batch.x.size(1)})")
print("\n")
print(f"2. GNN Embeddings (h): {node_embeddings.shape}")
print(f"   (Every node now has a {node_embeddings.size(1)}-dim structural summary)")
print("\n")
print(f"3. MLP Input (Concat):")
print(f"   Batch contains {len(example_edges)} edges.")
print(f"   Each edge vector is {2*node_embeddings.size(1)} dimensions (node_u + node_v).")
print("\n")
print(f"4. Final Output Distributions:")
print(f"   Classifier Probs (Min/Max): {probs.min().item():.4f} / {probs.max().item():.4f}")
print(f"   Ranking Scores (Sample):    {scores[:3].tolist()}")
print("\n")

In [ ]:
def train_and_evaluate(train_loader, val_loader, epochs=50):
    device = torch.device('cpu')
    
    # Initialize model and move to device
    model = StressMinimizationGNN(in_channels=6).to(device)
    
    # Reduced LR for stability and added weight_decay to prevent overfitting
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
    
    loss_fn_cls = nn.BCELoss()
    loss_fn_rank = RankingLoss(margin=1.0) 
    
    history = {'train_loss': [], 'val_loss': []}
    print(f"Starting Training on {device}...")

    for epoch in range(epochs):
        #  TRAINING PHASE
        model.train()
        total_train_loss = 0
        
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            
            # Extract targets from our 4-column y matrix
            target_edges = batch.y[:, 0:2]
            labels = batch.y[:, 2]
            gt_reductions = batch.y[:, 3]
            
            # Forward pass
            pred_probs, pred_scores, _ = model(batch.x, batch.edge_index, target_edges)
            
            # 1. Classification Loss (Is the edge good?)
            l_cls = loss_fn_cls(pred_probs, labels)
            
            # 2. Ranking Loss (Which edge is better?)
            # log1p is used because stress reduction values have a very high range
            gt_log = torch.log1p(gt_reductions.clamp(min=0)) 
            l_rank = loss_fn_rank(pred_scores, gt_log)
            
            # Combine losses
            combined_loss = l_cls + l_rank
            
            combined_loss.backward()
            optimizer.step()
            total_train_loss += combined_loss.item()
            
        #  VALIDATION PHASE
        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for v_batch in val_loader:
                v_batch = v_batch.to(device)
                v_edges = v_batch.y[:, 0:2]
                v_labels = v_batch.y[:, 2]
                v_gt = v_batch.y[:, 3]
                
                p_p, p_s, _ = model(v_batch.x, v_batch.edge_index, v_edges)
                
                v_gt_log = torch.log1p(v_gt.clamp(min=0))
                v_loss = loss_fn_cls(p_p, v_labels) + loss_fn_rank(p_s, v_gt_log)
                total_val_loss += v_loss.item()

        # Save statistics
        avg_train = total_train_loss / len(train_loader)
        avg_val = total_val_loss / len(val_loader)
        history['train_loss'].append(avg_train)
        history['val_loss'].append(avg_val)
        
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch:03d} | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f}")
            
    return model, history

trained_model, train_history = train_and_evaluate(train_loader, val_loader, epochs=100)

In [ ]:
model_save_path = os.path.join(DATA_FOLDER, 'stress_gnn_model_v1.pth')

# Save the weights
torch.save(trained_model.state_dict(), model_save_path)

print(f"Model weights saved successfully to: {model_save_path}")

In [ ]:
plt.figure(figsize=(10, 6))

# Plot Training Loss
plt.plot(train_history['train_loss'], 
         label='Training Loss (Total)', 
         color='#1f77b4', 
         linewidth=2, 
         alpha=0.8)

# Plot Validation Loss
plt.plot(train_history['val_loss'], 
         label='Validation Loss (Total)', 
         color='#d62728', 
         linestyle='--', 
         linewidth=2, 
         alpha=0.9)


plt.title('GNN Training Progress: Stress Centrality Reduction Task', fontsize=15, pad=15)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Loss (BCE + Pairwise Ranking)', fontsize=12)
plt.legend(loc='upper right', fontsize=11)
plt.grid(True, which="both", linestyle='--', alpha=0.5)

plt.show()


In [ ]:
import numpy as np

def get_ap_score(pred_probs, pred_scores, gt_values, k=10):
    
    # Convert to numpy if they are still torch tensors
    if hasattr(pred_probs, 'detach'): pred_probs = pred_probs.detach().cpu().numpy()
    if hasattr(pred_scores, 'detach'): pred_scores = pred_scores.detach().cpu().numpy()
    if hasattr(gt_values, 'detach'): gt_values = gt_values.detach().cpu().numpy()

    # Heuristic: Only consider edges the model is confident in (prob > 0.5)
    # We set scores of "bad" edges to a very low number so they don't appear in top k
    filtered_scores = np.where(pred_probs > 0.5, pred_scores, -1e9)
    
    # Get the top k indices of our model's predictions
    # If the model found fewer than k positive edges, it will fill with the best of the 'bad' ones
    pred_indices = np.argsort(filtered_scores)[::-1][:k]
    
    # Get the indices of the actual top k best edges (real)
    gt_indices = set(np.argsort(gt_values)[::-1][:k])
    
    relevant_found = 0
    precision_sum = 0
    
    # Calculate Precision at each rank i
    for i, idx in enumerate(pred_indices):
        if idx in gt_indices:
            relevant_found += 1
            # Precision at this position = (number of relevant items found so far) / (current rank)
            precision_at_i = relevant_found / (i + 1)
            precision_sum += precision_at_i
            
    # The Average Precision is the sum of precisions divided by k
    return precision_sum / k


In [ ]:
def final_test_performance(model, test_graphs_dict, k=100):
    
    model.eval()
    # Detect if model is on CUDA or CPU automatically
    device = next(model.parameters()).device
    results = []

    print("\n")
    print(f" INDUCTIVE TEST PERFORMANCE (mAP@{k}) ")
    print("\n")
    print(f"{'Graph Name':<12} | {'AP@'+str(k):<10}")
    print("\n")

    for key in test_graphs_dict:
        # Get the networkx graph object
        G = test_graphs_dict[key]
        
        # Extract features and topology
        node_x = features(G).to(device)
        edge_index = torch.tensor(list(G.edges), dtype=torch.long).T.to(device)
        
        # Load the pre-processed test data
        _, _, edge_labels, edge_regress = load_files(key)
        
        # Prepare candidate edges for prediction
        all_test_edges = edge_labels[1] + edge_labels[0]
        edges_t = torch.tensor(all_test_edges, dtype=torch.float).to(device)
        gt_vals = np.array(edge_regress)
        
        # Model Inference
        with torch.no_grad():
            # Forward returns (probs, scores, _)
            probs, scores, _ = model(node_x, edge_index, edges_t)
            
        # Calculate Average Precision for THIS graph
        ap_at_k = get_map_score(probs.cpu().numpy(), scores.cpu().numpy(), gt_vals, k=k)
        results.append(ap_at_k)
        
        print(f"{key:<12} | {ap_at_k:.4f}")

    # Final Mean Average Precision
    overall_map = np.mean(results)
    print("-" * 25)
    print(f"OVERALL Mean mAP@{k}: {overall_map:.4f}")
    print("="*45)
    
    return overall_map

mAP_100 = final_test_performance(trained_model, test_set, k=100)

In [ ]:
def suggest_best_edges(model,G,top_n=5):
    
    model.eval()
    
    node_features=features(G) 
    edge_index=torch.tensor(list(G.edges),dtype=torch.long).T
    
    # All possible new edges
    non_edges=list(nx.non_edges(G))
    edge_list=torch.tensor(non_edges,dtype=torch.float)
    
    # Model Predictions
    with torch.no_grad():
        probs,scores,_=model(node_features,edge_index,edge_list)
    
    probs_np=probs.numpy()   # shape: (num_non_edges,)
    scores_np=scores.numpy() # shape: (num_non_edges,)
    
    # Start with -inf for all edges
    combined=np.full(len(non_edges),-np.inf)
    
    # Only rank edges that model classifies as positive (prob > 0.5)
    pos_mask=probs_np>0.5
    
    # if model finds NO positive edges at all,just use all edges ranked by probability alone
    if pos_mask.sum()==0:
        print("Warning: No edges classified as positive. Falling back to prob ranking.")
        combined=probs_np.copy()
    else:
        # Among positive edges,rank by predicted score
        combined[pos_mask]=scores_np[pos_mask]
    
    # Get top_n indices
    top_indices=np.argsort(combined)[::-1][:top_n]
    
    suggestions=[]
    for idx in top_indices:
        suggestions.append({
            'edge':non_edges[idx],
            'probability_good':round(float(probs_np[idx]),4),
            'predicted_reduction':round(float(scores_np[idx]),4)
        })
    
    return suggestions

In [ ]:
test_graph_key = 'ba3' 
G_test = test_set[test_graph_key]

# Model Suggestions
print(f"\nModel Suggestions for {test_graph_key}\n")
recommendations=suggest_best_edges(trained_model,G_test,top_n=5)

edge_table,sorted_actual,_,_ =load_files(test_graph_key)

# Comparison
for i,rec in enumerate(recommendations):
    #the real reduction for the edge the model suggested
    actual_val =edge_table.get(rec['edge'], "Not Found")
    
    print(f"Rank {i+1} Prediction: Edge {rec['edge']}")
    print(f">Model predicted reduction: {rec['predicted_reduction']}")
    print(f">Actual real reduction:    {actual_val}")
    print("\n")


# actual top 5 
print(f"\n--- Actual Top 5 Best Edges (Real) ---")
for i in range(5):
    edge,val=sorted_actual[i]
    print(f"True Rank {i+1}: Edge {edge} | Actual Reduction: {val}")